In [3]:
import os
from pinecone import Pinecone

# Load your environment variables
PINECONE_API_KEY = 
PINECONE_ENV = 
INDEX_NAME =   # The index you ingested into

# Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY, environment=PINECONE_ENV)

# List all indexes
print("Indexes available:", pc.list_indexes().names())

# Connect to your index
index = pc.Index(INDEX_NAME)

# Check the index stats
stats = index.describe_index_stats()
print("Index stats:", stats)

Indexes available: ['trailblaze-index', 'trailblazeai']
Index stats: {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '188',
                                    'content-type': 'application/json',
                                    'date': 'Fri, 13 Mar 2026 20:13:45 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '4',
                                    'x-pinecone-request-latency-ms': '4',
                                    'x-pinecone-response-duration-ms': '5'}},
 'dimension': 1536,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 1610}},
 'storageFullness': 0.0,
 'total_vector_count': 1610,
 'vector_type': 'dense'}


In [7]:
pc.delete_index(name="trailblaze-index")

In [4]:
import os
from dotenv import load_dotenv
from langchain_aws import BedrockEmbeddings
import pinecone

# Load environment variables
load_dotenv()
BEDROCK_EMBED_MODEL = os.environ.get("BEDROCK_EMBED_MODEL")
AWS_REGION = os.environ.get("AWS_REGION")
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
PINECONE_ENV = os.environ.get("PINECONE_ENV")

# sanity check
assert BEDROCK_EMBED_MODEL, "BEDROCK_EMBED_MODEL is not set in .env!"

# Initialize Pinecone
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY)
index_name = "trailblazeai"

# Initialize Bedrock embeddings
embedding_fn = BedrockEmbeddings(model_id=BEDROCK_EMBED_MODEL, region_name=AWS_REGION)

In [2]:
import os
from dotenv import load_dotenv
from langchain_aws import BedrockEmbeddings
import pinecone

load_dotenv()

BEDROCK_EMBED_MODEL = os.environ.get("BEDROCK_EMBED_MODEL")
AWS_REGION = os.environ.get("AWS_REGION")
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
PINECONE_ENV = os.environ.get("PINECONE_ENV")
INDEX_NAME = "trailblazeai"

# Initialize Pinecone client
pc = pinecone.Pinecone(api_key=PINECONE_API_KEY, environment=PINECONE_ENV)

# Initialize embeddings
embedding_fn = BedrockEmbeddings(model_id=BEDROCK_EMBED_MODEL, region_name=AWS_REGION)

# Create Index object
index = pc.Index(INDEX_NAME)

def query_pinecone(query, top_k=5):
    query_vector = embedding_fn.embed_query(query)

    # Correct way: call query on the Index object
    response = index.query(
        vector=query_vector,
        top_k=top_k,
        include_metadata=True
    )

    results = [match['metadata']['text'] for match in response['matches']]
    return results

# Example usage
question = "Summarize the key points to a 9 year old on what it says"
matches = query_pinecone(question)

for i, chunk in enumerate(matches, 1):
    print(f"{i}. {chunk}\n")

KeyError: 'text'